# Feast offline-to-online training and inference

This notebook runs a production-shaped, CPU-only feature pipeline:

1. a Spark Operator `SparkApplication` generates a synthetic batch dataset and stores it as partitioned Parquet in S3-compatible object storage;
2. Spark performs Feast's point-in-time historical join and trains a small NumPy model;
3. Feast uses PostgreSQL for its durable SQL registry and materializes the latest feature values into Redis; and
4. KServe deploys a model server that reads Redis-backed online features before predicting.

Object storage is the durable, versionable offline data layer; operator-managed Spark provides elastic batch compute; PostgreSQL stores Feast metadata rather than feature history; and Redis serves low-latency online lookups. Feast classifies its Spark offline store as a contributed integration without full test coverage, so qualify it against your scale and upgrade requirements or use a fully supported warehouse while retaining the same S3-and-Spark data pipeline.

The reusable source code and Kubernetes manifests are in `assets/feast-offline-to-online-inference/`. The command cells below assume that path is relative to the notebook working directory. Set `FEAST_ASSET_DIR` when your Workbench starts in a different directory.

Prerequisites:

- the Alauda Spark Operator is installed through OLM and `sparkapplications.sparkoperator.k8s.io` exists;
- `FEAST_SPARK_IMAGE` is set to a Spark runtime containing PySpark, Feast 0.61.x with Spark and Redis support, NumPy, pandas, PyArrow, PyYAML, and a Hadoop S3A connector compatible with the image's Hadoop version;
- a `feast-data-stores` Secret in `feast-demo` with `redis` and `sql` keys;
- a `feast-s3-credentials` Secret in `feast-demo` with `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_ENDPOINT_URL`, and `S3_BUCKET`;
- a pre-created S3 bucket, a default ReadWriteOnce storage class, and the Feast and KServe operators; and
- `bash`, `kubectl`, `sed`, and `curl` in the Workbench image.

Find the global-cluster registry without copying an internal registry hostname from this document:

```bash
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
```

Choose an approved Spark runtime from that registry and set its complete reference in `FEAST_SPARK_IMAGE`. Optionally set `FEAST_SPARK_VERSION` and `FEAST_MODEL_IMAGE`; the commands default to Spark 4.0.1 metadata and the Feast 0.61.0 model-server repository in the discovered registry.


## 1. Verify the operator and asset bundle

The Spark runtime image contains Spark and its S3A client libraries. No separate Spark or Hadoop service is installed by this example.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
test -f "$ASSET_DIR/batch.py"
test -f "$ASSET_DIR/spark-application.yaml"
: "${FEAST_SPARK_IMAGE:?Set FEAST_SPARK_IMAGE to an approved global-registry image}"
kubectl get crd sparkapplications.sparkoperator.k8s.io
kubectl get crd featurestores.feast.dev
kubectl get crd inferenceservices.serving.kserve.io
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
echo


## 2. Prepare PostgreSQL registry and Redis online serving

The Feast Operator manages the control plane and online-serving backends. PostgreSQL stores the SQL registry and Redis stores materialized online feature values. Spark reads the offline Parquet history directly from S3.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
kubectl apply -f "$ASSET_DIR/namespaces.yaml"
kubectl -n feast-demo get secret feast-data-stores
kubectl -n feast-demo get secret feast-s3-credentials
kubectl apply -f "$ASSET_DIR/feature-store.yaml"

for _ in $(seq 1 60); do
  phase="$(kubectl -n feast-demo get featurestore feast-notebook \
    -o jsonpath='{.status.phase}' 2>/dev/null || true)"
  echo "${phase:-Pending}"
  [ "$phase" = Ready ] && break
  if [ "$phase" = Failed ]; then
    kubectl -n feast-demo describe featurestore feast-notebook
    exit 1
  fi
  sleep 10
done
[ "${phase:-}" = Ready ] || { echo "FeatureStore did not become Ready" >&2; exit 1; }
kubectl -n feast-demo get featurestore feast-notebook


## 3. Submit the synthetic S3 batch as a SparkApplication

The code ConfigMap packages `batch.py` and `server.py` from the asset directory. The Spark driver configures S3A through the public `SparkSession.builder.config()` API before creating the Spark context. S3A reads credentials from the Secret-backed driver and executor environments, so the keys are not written into the `SparkApplication` or model PVC.

The application generates deterministic synthetic driver events, writes partitioned Parquet to S3, registers a Feast `SparkSource`, performs the historical join, trains the sample model, and materializes the same features into Redis.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
: "${FEAST_SPARK_IMAGE:?Set FEAST_SPARK_IMAGE to an approved global-registry image}"
SPARK_VERSION="${FEAST_SPARK_VERSION:-4.0.1}"

kubectl -n feast-demo create configmap feast-offline-batch-code \
  --from-file=batch.py="$ASSET_DIR/batch.py" \
  --from-file=server.py="$ASSET_DIR/server.py" \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl apply -f "$ASSET_DIR/spark-rbac.yaml"
kubectl apply -f "$ASSET_DIR/model-pvc.yaml"
kubectl -n feast-demo delete sparkapplication feast-offline-batch \
  --ignore-not-found --wait=true
sed \
  -e "s|FEAST_SPARK_IMAGE_PLACEHOLDER|$FEAST_SPARK_IMAGE|g" \
  -e "s|FEAST_SPARK_VERSION_PLACEHOLDER|$SPARK_VERSION|g" \
  "$ASSET_DIR/spark-application.yaml" | kubectl apply -f -

state=""
for _ in $(seq 1 120); do
  state="$(kubectl -n feast-demo get sparkapplication feast-offline-batch \
    -o jsonpath='{.status.applicationState.state}' 2>/dev/null || true)"
  echo "${state:-SUBMITTED}"
  case "$state" in
    COMPLETED|FAILED|FAILED_SUBMISSION|INVALIDATING|UNKNOWN) break ;;
  esac
  sleep 10
done
driver="$(kubectl -n feast-demo get sparkapplication feast-offline-batch \
  -o jsonpath='{.status.driverInfo.podName}' 2>/dev/null || true)"
if [ "$state" != COMPLETED ]; then
  [ -n "$driver" ] && kubectl -n feast-demo logs "$driver" --tail=300 || true
  exit 1
fi
[ -n "$driver" ] && kubectl -n feast-demo logs "$driver" --tail=100


## 4. Inspect the offline-to-online result

The completed Spark driver writes a small online-feature verification sample and the model artifact to the PVC. This short-lived inspector reads the sample without embedding it in the notebook.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
MODEL_IMAGE="${FEAST_MODEL_IMAGE:-}"
if [ -z "$MODEL_IMAGE" ]; then
  registry="$(kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}')"
  MODEL_IMAGE="$registry/mlops/feast/feature-server:0.61.0"
fi
kubectl -n feast-demo delete pod feast-model-inspector --ignore-not-found --wait=true
sed "s|FEAST_MODEL_IMAGE_PLACEHOLDER|$MODEL_IMAGE|g" \
  "$ASSET_DIR/model-inspector.yaml" | kubectl apply -f -
kubectl -n feast-demo wait \
  "--for=jsonpath={.status.phase}=Succeeded" pod/feast-model-inspector --timeout=180s
kubectl -n feast-demo logs feast-model-inspector
kubectl -n feast-demo delete pod feast-model-inspector --wait=true


## 5. Deploy the Redis-backed model with KServe

The model server loads the weights and serving-only Feast configuration from the PVC, queries Redis through the operator-managed Feast online service, and exposes KServe's v2 inference protocol.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
MODEL_IMAGE="${FEAST_MODEL_IMAGE:-}"
if [ -z "$MODEL_IMAGE" ]; then
  registry="$(kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}')"
  MODEL_IMAGE="$registry/mlops/feast/feature-server:0.61.0"
fi
sed "s|FEAST_MODEL_IMAGE_PLACEHOLDER|$MODEL_IMAGE|g" \
  "$ASSET_DIR/serving-runtime.yaml" | kubectl apply -f -
kubectl apply -f "$ASSET_DIR/inference-service.yaml"
kubectl -n feast-demo get inferenceservice feast-online-model


## 6. Send an online-feature prediction

After the predictor has an available replica, send driver IDs to the KServe v2 endpoint. The server retrieves their materialized features from Redis and combines those values with the model trained by the `SparkApplication`.


In [ ]:
%%bash
set -euo pipefail
available=""
for _ in $(seq 1 90); do
  available="$(kubectl -n feast-demo get deployment feast-online-model-predictor \
    -o jsonpath='{.status.availableReplicas}' 2>/dev/null || true)"
  echo "availableReplicas=${available:-0}"
  [ "${available:-0}" -ge 1 ] 2>/dev/null && break
  sleep 10
done
[ "${available:-0}" -ge 1 ] 2>/dev/null || exit 1

kubectl -n feast-demo port-forward service/feast-online-model-predictor 18080:80 \
  >/tmp/feast-model-port-forward.log 2>&1 &
port_forward_pid=$!
trap 'kill "$port_forward_pid" 2>/dev/null || true' EXIT
for _ in $(seq 1 30); do
  curl -fsS http://127.0.0.1:18080/v2/health/ready >/dev/null 2>&1 && break
  sleep 2
done
curl -fsS -X POST \
  http://127.0.0.1:18080/v2/models/feast-online-model/infer \
  -H 'Content-Type: application/json' \
  -d '{"inputs":[{"name":"driver_id","shape":[2],"datatype":"INT64","data":[1,2]}]}'
echo
